In [ ]:
import pandas as pd
from pathlib import Path
import re
import time

# ==================================================
# FOLDER
# ==================================================

folder = Path("degree_year")

# Output files
output_detail = "IPEDS_degree_clean_all_rows.csv"
output_summary = "IPEDS_degree_summary_clean.csv"

# ==================================================
# AWARD LEVEL NAMES
# ==================================================

award_level_map = {
    1: "Certificate < 1 year",
    2: "Certificate 1-2 years",
    3: "Associate",
    4: "Certificate 2-4 years",
    5: "Bachelor",
    6: "Post-baccalaureate certificate",
    7: "Master",
    8: "Post-master certificate",
    9: "Doctor",
    10: "First-professional degree",
    11: "First-professional certificate",
    17: "Doctor - research/scholarship",
    18: "Doctor - professional practice",
    19: "Doctor - other",
}

def degree_group(x):
    if pd.isna(x):
        return "Unknown"

    x = int(x)

    if x == 3:
        return "Associate"
    elif x == 5:
        return "Bachelor"
    elif x == 7:
        return "Master"
    elif x in [9, 17, 18, 19]:
        return "Doctor"
    elif x in [10, 11]:
        return "Professional"
    elif x in [1, 2, 4, 6, 8]:
        return "Certificate"
    else:
        return "Other"

# ==================================================
# CLEAN CIP CODE
# Example:
# 10102   -> 01.0102
# 6.0101  -> 06.0101
# ==================================================

def clean_cip_code(value):
    if pd.isna(value):
        return pd.NA

    s = str(value).strip().replace('"', "")

    if s == "" or s.lower() in ["nan", "none"]:
        return pd.NA

    if "." in s:
        left, right = s.split(".", 1)
        left = re.sub(r"\D", "", left).zfill(2)
        right = re.sub(r"\D", "", right).ljust(4, "0")[:4]
        return f"{left}.{right}"

    digits = re.sub(r"\D", "", s)

    if digits == "":
        return pd.NA

    digits = digits.zfill(6)[-6:]
    return f"{digits[:2]}.{digits[2:]}"

# ==================================================
# GET YEAR FROM FILE NAME
# Example:
# 1984_c1984_cip.csv -> 1984
# ==================================================

def extract_year(filename):
    match = re.search(r"(19|20)\d{2}", filename)
    if match:
        return int(match.group(0))
    return pd.NA

# ==================================================
# READ ONE FILE
# ==================================================

needed_cols = {
    "unitid",
    "cipcode",
    "awlevel",
    "crace15",
    "crace16",
    "majornum",
    "part"
}

def read_one_file(file_path):
    # Read only columns we need
    try:
        df = pd.read_csv(
            file_path,
            dtype=str,
            low_memory=False,
            usecols=lambda c: c.strip().lower() in needed_cols
        )
    except UnicodeDecodeError:
        df = pd.read_csv(
            file_path,
            dtype=str,
            low_memory=False,
            encoding="latin1",
            usecols=lambda c: c.strip().lower() in needed_cols
        )

    # Clean column names
    df.columns = df.columns.str.lower().str.strip()

    # Check required columns
    required = {"unitid", "cipcode", "awlevel", "crace15", "crace16"}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(f"Missing columns: {missing}")

    # Add year and source file
    df["year"] = extract_year(file_path.name)
    df["source_file"] = file_path.name

    # Clean CIP
    df["cipcode_raw"] = df["cipcode"]
    df["cip_code"] = df["cipcode"].apply(clean_cip_code)

    # Clean award level
    df["awlevel"] = pd.to_numeric(df["awlevel"], errors="coerce").astype("Int64")
    df["degree_level"] = df["awlevel"].map(award_level_map).fillna("Unknown")
    df["degree_group"] = df["awlevel"].apply(degree_group)

    # IMPORTANT:
    # crace15 = total men
    # crace16 = total women
    # Do NOT sum all crace columns because that can double count.
    df["men_count"] = pd.to_numeric(df["crace15"], errors="coerce").fillna(0).astype(int)
    df["women_count"] = pd.to_numeric(df["crace16"], errors="coerce").fillna(0).astype(int)
    df["degree_count"] = df["men_count"] + df["women_count"]

    # Some years do not have these columns
    if "majornum" not in df.columns:
        df["majornum"] = pd.NA

    if "part" not in df.columns:
        df["part"] = pd.NA

    # Keep clean columns only
    clean_df = df[
        [
            "year",
            "unitid",
            "cip_code",
            "cipcode_raw",
            "awlevel",
            "degree_level",
            "degree_group",
            "men_count",
            "women_count",
            "degree_count",
            "majornum",
            "part",
            "source_file"
        ]
    ].copy()

    # Data cleaning
    clean_df = clean_df.dropna(subset=["year", "unitid", "cip_code", "awlevel"])
    clean_df = clean_df[clean_df["degree_count"] > 0]
    clean_df = clean_df.drop_duplicates()

    return clean_df

# ==================================================
# READ ALL CSV FILES
# ==================================================

files = sorted(folder.glob("*.csv"))

if len(files) == 0:
    raise FileNotFoundError("No CSV files found in folder: degree_year")

print(f"Found {len(files)} CSV files in folder: {folder}")

all_data = []
errors = []

start_time = time.time()

for i, file_path in enumerate(files, start=1):
    try:
        clean_file = read_one_file(file_path)
        all_data.append(clean_file)
    except Exception as e:
        errors.append({
            "file": file_path.name,
            "error": str(e)
        })

    elapsed = time.time() - start_time
    avg_time = elapsed / i
    remaining = avg_time * (len(files) - i)

    print(
        f"Loading all files: {i}/{len(files)} "
        f"({i / len(files):.0%}) | "
        f"Elapsed: {elapsed:.1f}s | "
        f"Left: {remaining:.1f}s",
        end="\r"
    )

print("\nFinished reading files.")

# ==================================================
# COMBINE ALL FILES
# ==================================================

combined = pd.concat(all_data, ignore_index=True)

combined = combined.sort_values(
    by=["year", "unitid", "cip_code", "awlevel"]
).reset_index(drop=True)

# Save detailed clean file
combined.to_csv(output_detail, index=False)

# ==================================================
# MAKE SUMMARY FILE
# Total degree count by year, CIP, and degree level
# ==================================================

summary = (
    combined
    .groupby(
        ["year", "cip_code", "awlevel", "degree_level", "degree_group"],
        as_index=False
    )
    .agg(
        degree_count=("degree_count", "sum"),
        men_count=("men_count", "sum"),
        women_count=("women_count", "sum"),
        institutions=("unitid", "nunique"),
        rows=("unitid", "size")
    )
)

summary = summary.sort_values(
    by=["year", "cip_code", "awlevel"]
).reset_index(drop=True)

summary.to_csv(output_summary, index=False)

# ==================================================
# PRINT RESULTS
# ==================================================

print("Clean detailed file saved as:", output_detail)
print("Clean summary file saved as:", output_summary)

print("\nCombined rows:", len(combined))
print("Summary rows:", len(summary))
print("Year range:", combined["year"].min(), "-", combined["year"].max())

if errors:
    print("\nSome files had errors:")
    for e in errors:
        print(e["file"], "->", e["error"])
else:
    print("\nNo file errors.")

print("\nPreview of clean summary:")
print(summary.head(20))

In [ ]:
import pandas as pd
from pathlib import Path
import re
import time

# ==================================================
# FOLDER NAME
# ==================================================

folder = Path("degree_year")

# Output file with ALL original columns + clean columns
output_file = "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv"

# ==================================================
# AWARD LEVEL MAP
# ==================================================

award_level_map = {
    1: "Certificate < 1 year",
    2: "Certificate 1-2 years",
    3: "Associate",
    4: "Certificate 2-4 years",
    5: "Bachelor",
    6: "Post-baccalaureate certificate",
    7: "Master",
    8: "Post-master certificate",
    9: "Doctor",
    10: "First-professional degree",
    11: "First-professional certificate",
    17: "Doctor - research/scholarship",
    18: "Doctor - professional practice",
    19: "Doctor - other",
}

def degree_group(x):
    if pd.isna(x):
        return "Unknown"

    x = int(x)

    if x == 3:
        return "Associate"
    elif x == 5:
        return "Bachelor"
    elif x == 7:
        return "Master"
    elif x in [9, 17, 18, 19]:
        return "Doctor"
    elif x in [10, 11]:
        return "Professional"
    elif x in [1, 2, 4, 6, 8]:
        return "Certificate"
    else:
        return "Other"

# ==================================================
# CLEAN CIP CODE
# ==================================================

def clean_cip_code(value):
    if pd.isna(value):
        return pd.NA

    s = str(value).strip().replace('"', "")

    if s == "" or s.lower() in ["nan", "none"]:
        return pd.NA

    if "." in s:
        left, right = s.split(".", 1)
        left = re.sub(r"\D", "", left).zfill(2)
        right = re.sub(r"\D", "", right).ljust(4, "0")[:4]
        return f"{left}.{right}"

    digits = re.sub(r"\D", "", s)

    if digits == "":
        return pd.NA

    digits = digits.zfill(6)[-6:]
    return f"{digits[:2]}.{digits[2:]}"

# ==================================================
# GET YEAR FROM FILE NAME
# ==================================================

def extract_year(filename):
    match = re.search(r"(19|20)\d{2}", filename)
    if match:
        return int(match.group(0))
    return pd.NA

# ==================================================
# READ CSV SAFELY
# ==================================================

def read_csv_safely(file_path):
    try:
        return pd.read_csv(file_path, dtype=str, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(file_path, dtype=str, low_memory=False, encoding="latin1")

# ==================================================
# READ ALL FILES AND KEEP ALL COLUMNS
# ==================================================

files = sorted(folder.glob("*.csv"))

if len(files) == 0:
    raise FileNotFoundError("No CSV files found inside folder: degree_year")

print(f"Found {len(files)} CSV files")

all_data = []
errors = []

start_time = time.time()

for i, file_path in enumerate(files, start=1):
    try:
        df = read_csv_safely(file_path)

        # Clean column names
        df.columns = df.columns.str.lower().str.strip()

        # Add year and source file
        df["year"] = extract_year(file_path.name)
        df["source_file"] = file_path.name

        # Add clean CIP code if cipcode exists
        if "cipcode" in df.columns:
            df["cip_code_clean"] = df["cipcode"].apply(clean_cip_code)
        else:
            df["cip_code_clean"] = pd.NA

        # Add degree level if awlevel exists
        if "awlevel" in df.columns:
            df["awlevel_clean"] = pd.to_numeric(df["awlevel"], errors="coerce").astype("Int64")
            df["degree_level"] = df["awlevel_clean"].map(award_level_map).fillna("Unknown")
            df["degree_group"] = df["awlevel_clean"].apply(degree_group)
        else:
            df["awlevel_clean"] = pd.NA
            df["degree_level"] = "Unknown"
            df["degree_group"] = "Unknown"

        # Add degree count if crace15 and crace16 exist
        if "crace15" in df.columns and "crace16" in df.columns:
            df["men_count"] = pd.to_numeric(df["crace15"], errors="coerce").fillna(0).astype(int)
            df["women_count"] = pd.to_numeric(df["crace16"], errors="coerce").fillna(0).astype(int)
            df["degree_count"] = df["men_count"] + df["women_count"]
        else:
            df["men_count"] = 0
            df["women_count"] = 0
            df["degree_count"] = 0

        all_data.append(df)

    except Exception as e:
        errors.append({
            "file": file_path.name,
            "error": str(e)
        })

    elapsed = time.time() - start_time
    avg_time = elapsed / i
    remaining = avg_time * (len(files) - i)

    print(
        f"Loading: {i}/{len(files)} "
        f"({i / len(files):.0%}) | "
        f"Elapsed: {elapsed:.1f}s | "
        f"Left: {remaining:.1f}s",
        end="\r"
    )

print("\nFinished reading all files.")

# ==================================================
# COMBINE EVERYTHING
# ==================================================

all_info = pd.concat(all_data, ignore_index=True, sort=False)

# Optional: sort if these columns exist
sort_cols = [col for col in ["year", "unitid", "cip_code_clean", "awlevel_clean"] if col in all_info.columns]

if sort_cols:
    all_info = all_info.sort_values(by=sort_cols).reset_index(drop=True)

# Save file
all_info.to_csv(output_file, index=False)

# ==================================================
# PRINT RESULT
# ==================================================

print("\nSaved file:", output_file)
print("Total rows:", len(all_info))
print("Total columns:", len(all_info.columns))

print("\nColumns:")
for col in all_info.columns:
    print(col)

print("\nPreview:")
print(all_info.head(10))

if errors:
    print("\nFiles with errors:")
    for e in errors:
        print(e["file"], "->", e["error"])
else:
    print("\nNo file errors.")

In [7]:
# IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv

In [ ]:
df = pd.read_csv("IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv")

/var/folders/yz/w04hn7v511502rh_pt_lfckm0000gn/T/ipykernel_57820/1894267249.py:1: DtypeWarning: Columns (14,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,83,85,87,89,91,93,95,98,100,102,104,106,108,110,112,114,116,118,120,122,124,126,128,130,132,134,136,138,140,142,144,146,148,150,152,154,156,158,160,162,164,166,168,170,172,174,176,178,180,182,184,186,188) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv")


In [5]:
for col in df.columns:
    print(col)

unitid
cipcode
majornum
awlevel
xctotalt
ctotalt
xctotalm
ctotalm
xctotalw
ctotalw
xcaiant
caiant
xcaianm
caianm
xcaianw
caianw
xcasiat
casiat
xcasiam
casiam
xcasiaw
casiaw
xcbkaat
cbkaat
xcbkaam
cbkaam
xcbkaaw
cbkaaw
xchispt
chispt
xchispm
chispm
xchispw
chispw
xcnhpit
cnhpit
xcnhpim
cnhpim
xcnhpiw
cnhpiw
xcwhitt
cwhitt
xcwhitm
cwhitm
xcwhitw
cwhitw
xc2mort
c2mort
xc2morm
c2morm
xc2morw
c2morw
xcunknt
cunknt
xcunknm
cunknm
xcunknw
cunknw
xcnralt
cnralt
xcnralm
cnralm
xcnralw
cnralw
year
source_file
cip_code_clean
awlevel_clean
degree_level
degree_group
men_count
women_count
degree_count


In [6]:
print(df.head(10))

   unitid  cipcode majornum awlevel xctotalt ctotalt xctotalm ctotalm  \
0  100654  01.0999        1       5        R      12        R       2   
1  100654  01.1001        1       5        R      10        R       1   
2  100654  01.1001        1       7        R       7        R       1   
3  100654  01.1001        1      17        R       6        R       3   
4  100654  01.9999        1       5        R       2        R       1   
5  100654  01.9999        1       7        R       9        R       4   
6  100654  01.9999        1      17        R       3        R       0   
7  100654  03.0599        1       5        R       0        R       0   
8  100654  04.0301        1       5        R       8        R       1   
9  100654  04.0301        1       7        R      10        R       5   

  xctotalw ctotalw  ... cnralw  year       source_file cip_code_clean  \
0        R      10  ...      0  2024  2024_c2024_a.csv        01.0999   
1        R       9  ...      0  2024  2024_c2024_a